
### 02 — Text Cleaning and Tokenization

##### Purpose

This notebook prepares raw support-ticket text for downstream NLP feature engineering and classification.

It:

- reads the Bronze support-ticket table
- validates the source schema
- performs basic data-quality checks
- applies reusable text-cleaning logic
- tokenizes cleaned text
- validates the transformed dataset
- persists the curated output to the Silver table

The notebook is independently runnable.

It does not depend on another notebook or %run.

Shared configuration comes from: src/project_config.py

Reusable text-cleaning logic comes from: src/text_preprocessing.py

##### 1. Architecture

``` text

bronze_support_tickets
        ↓
Schema Validation
        ↓
Data Quality Validation
        ↓
Text Cleaning
        ↓
Tokenization
        ↓
Transformation Validation
        ↓
nlp_preprocessed_tickets

```

Project responsibilities:

``` text

src/project_config.py
        ↓
stable project-wide configuration

src/text_preprocessing.py
        ↓
reusable text-cleaning logic

02_text_cleaning_and_tokenization
        ↓
orchestration
validation
inspection
persistence

```


##### 2. Technologies

This notebook uses:

- Python
- PySpark
- Regular Expressions
- Delta Lake
- Unity Catalog
- Databricks

##### 3. Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from src.project_config import (
    SOURCE_TABLE,
    NLP_CLEAN_TABLE,
    TICKET_ID_COL,
    TEXT_COL,
    TARGET_COL,
    CLEAN_TEXT_COL,
    TOKENS_COL,
    TOKEN_COUNT_COL,
    EXPECTED_CATEGORIES,
)

from src.text_preprocessing import clean_text_column

##### 4. Verify Project Configuration

In [0]:
print(f"Source table : {SOURCE_TABLE}")
print(f"Output table : {NLP_CLEAN_TABLE}")

print(f"Ticket ID    : {TICKET_ID_COL}")
print(f"Text column  : {TEXT_COL}")
print(f"Target       : {TARGET_COL}")

##### 5. Load Bronze Support Tickets

In [0]:
bronze_df = spark.table(SOURCE_TABLE)

In [0]:
display(
    bronze_df.limit(10)
)

In [0]:
bronze_df.printSchema()

In [0]:
source_row_count = bronze_df.count()

print(
    f"Bronze row count: "
    f"{source_row_count:,}"
)

In [0]:
if source_row_count == 0:
    raise ValueError(
        f"Source table contains no rows: "
        f"{SOURCE_TABLE}"
    )

##### 6. Validate Required Source Columns

###### The minimum input contract is:

- ticket_id
- ticket_text
- category

In [0]:
required_columns = {
    TICKET_ID_COL,
    TEXT_COL,
    TARGET_COL,
}

In [0]:
required_columns

In [0]:
available_columns = set(
    bronze_df.columns
)

missing_columns = (
    required_columns
    - available_columns
)

In [0]:
if missing_columns:
    raise ValueError(
        "Missing required columns in "
        f"{SOURCE_TABLE}: "
        f"{sorted(missing_columns)}"
    )

print(
    "Required column validation passed."
)

##### 7. Select the NLP Source Columns

In [0]:
tickets_df = (
    bronze_df
    .select(
        F.col(TICKET_ID_COL)
        .cast("string")
        .alias(TICKET_ID_COL),

        F.col(TEXT_COL)
        .cast("string")
        .alias(TEXT_COL),

        F.col(TARGET_COL)
        .cast("string")
        .alias(TARGET_COL),
    )
)

In [0]:
display(
    tickets_df.limit(10)
)

##### 8. Inspect Null Values

In [0]:
null_summary_df = (
    tickets_df
    .select(
        F.sum(
            F.col(TICKET_ID_COL)
            .isNull()
            .cast("int")
        ).alias(
            "null_ticket_id"
        ),

        F.sum(
            F.col(TEXT_COL)
            .isNull()
            .cast("int")
        ).alias(
            "null_ticket_text"
        ),

        F.sum(
            F.col(TARGET_COL)
            .isNull()
            .cast("int")
        ).alias(
            "null_category"
        ),
    )
)

In [0]:
display(
    null_summary_df
)

##### 9. Normalize Source Whitespace

In [0]:
#Normalize whitespace in the source text and target label.

tickets_df = (
    tickets_df
    .withColumn(
        TEXT_COL,
        F.trim(
            F.regexp_replace(
                F.col(TEXT_COL),
                r"\s+",
                " ",
            )
        ),
    )
    .withColumn(
        TARGET_COL,
        F.trim(
            F.col(TARGET_COL)
        ),
    )
)

##### 10. Remove Invalid Supervised-Learning Records

In [0]:
valid_tickets_df = (
    tickets_df
    .filter(
        F.col(TICKET_ID_COL).isNotNull()
        & F.col(TEXT_COL).isNotNull()
        & F.col(TARGET_COL).isNotNull()
    )
    .filter(
        F.length(
            F.col(TEXT_COL)
        ) > 0
    )
    .filter(
        F.length(
            F.col(TARGET_COL)
        ) > 0
    )
)

In [0]:
valid_row_count = (
    valid_tickets_df.count()
)

removed_row_count = (
    source_row_count
    - valid_row_count
)

print(
    f"Source rows : {source_row_count:,}"
)

print(
    f"Valid rows  : {valid_row_count:,}"
)

print(
    f"Removed rows: {removed_row_count:,}"
)

##### 11. Validate Ticket Categories

In [0]:
category_distribution_df = (
    valid_tickets_df
    .groupBy(
        TARGET_COL
    )
    .count()
    .orderBy(
        F.desc("count")
    )
)

In [0]:
display(
    category_distribution_df
)

In [0]:
EXPECTED_CATEGORIES = (
    "Billing",
    "Cancellation",
    "Login",
    "Technical",
)

In [0]:
actual_categories = {
    row[TARGET_COL]
    for row in (
        valid_tickets_df
        .select(TARGET_COL)
        .distinct()
        .collect()
    )
}

In [0]:
expected_categories = set(
    EXPECTED_CATEGORIES
)

unexpected_categories = (
    actual_categories
    - expected_categories
)

In [0]:
if unexpected_categories:
    raise ValueError(
        "Unexpected ticket categories found: "
        f"{sorted(unexpected_categories)}"
    )

print(
    "Ticket category validation passed."
)

##### 12. Check for Missing Expected Categories

In [0]:
#Unexpected labels are an error.Missing expected labels are worth detecting separately:


missing_categories = (
    expected_categories
    - actual_categories
)

if missing_categories:
    print(
        "Warning - expected categories "
        "not found in the current dataset:"
    )

    print(
        sorted(missing_categories)
    )
else:
    print(
        "All expected ticket categories "
        "are present."
    )

##### 13. Inspect Duplicate Ticket IDs

In [0]:
duplicate_ticket_ids_df = (
    valid_tickets_df
    .groupBy(
        TICKET_ID_COL
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

In [0]:
duplicate_ticket_id_count = (
    duplicate_ticket_ids_df.count()
)

print(
    "Duplicate ticket IDs: "
    f"{duplicate_ticket_id_count:,}"
)

In [0]:
display(
    duplicate_ticket_ids_df.limit(20)
)

We do not automatically drop duplicate IDs because duplicates may represent:

- duplicate ingestion
- multiple ticket messages
- ticket history
- source-system behavior

The correct treatment depends on business meaning.

##### 14. Inspect Duplicate Ticket Text

In [0]:
#Repeated text can create leakage later if identical records appear across train and test sets.

duplicate_text_df = (
    valid_tickets_df
    .groupBy(
        TEXT_COL,
        TARGET_COL,
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .orderBy(
        F.desc("count")
    )
)

In [0]:
display(
    duplicate_text_df.limit(20)
)

We inspect duplicates here.

Training-split leakage will be handled when we construct the model datasets.

##### 15. Text Cleaning Strategy

cleaning strategy is deliberately conservative.

perform:

- lowercasing
- URL normalization
- email normalization
- punctuation normalization
- whitespace normalization
- trimming

do not automatically perform:

- stop-word removal
- stemming
- lemmatization

For example:

- internet is working
- internet is not working

Removing not could destroy an important distinction.

##### 16. Reusable Text Cleaning Function

In [0]:
# It lives in: src/text_preprocessing.py

# The notebook imports:

from src.text_preprocessing import (
    clean_text_column,
)

##### 17. Apply Text Cleaning

In [0]:
clean_tickets_df = (
    valid_tickets_df
    .withColumn(
        CLEAN_TEXT_COL,
        clean_text_column(
            F.col(TEXT_COL)
        ),
    )
)

In [0]:
display(
    clean_tickets_df
    .select(
        TICKET_ID_COL,
        TEXT_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .limit(20)
)

##### 18. Remove Text Empty After Cleaning

In [0]:
clean_tickets_df = (
    clean_tickets_df
    .filter(
        F.length(
            F.col(CLEAN_TEXT_COL)
        ) > 0
    )
)

In [0]:
clean_row_count = (
    clean_tickets_df.count()
)

print(
    "Rows after text cleaning: "
    f"{clean_row_count:,}"
)

In [0]:
if clean_row_count == 0:
    raise ValueError(
        "No valid support-ticket records "
        "remain after text cleaning."
    )

##### 19. Tokenize the Cleaned Text

In [0]:
#For our traditional NLP baseline, use whitespace tokenization after cleaning.

clean_tickets_df = (
    clean_tickets_df
    .withColumn(
        TOKENS_COL,
        F.split(
            F.col(CLEAN_TEXT_COL),
            r"\s+",
        ),
    )
)

##### 20. Add Token Count

In [0]:
clean_tickets_df = (
    clean_tickets_df
    .withColumn(
        TOKEN_COUNT_COL,
        F.size(
            F.col(TOKENS_COL)
        ),
    )
)

In [0]:
display(
    clean_tickets_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TOKENS_COL,
        TOKEN_COUNT_COL,
        TARGET_COL,
    )
    .limit(20)
)

##### 21. Validate Tokenization

In [0]:
invalid_token_count = (
    clean_tickets_df
    .filter(
        F.col(TOKEN_COUNT_COL) <= 0
    )
    .count()
)

In [0]:
if invalid_token_count > 0:
    raise ValueError(
        f"Found {invalid_token_count} "
        "records with zero tokens."
    )

print(
    "Tokenization validation passed."
)

##### 22. Analyze Ticket Length

In [0]:
# Token count gives us a useful first view of document length.
ticket_length_summary_df = (
    clean_tickets_df
    .select(
        F.min(
            TOKEN_COUNT_COL
        ).alias(
            "min_tokens"
        ),

        F.expr(
            f"percentile_approx("
            f"{TOKEN_COUNT_COL}, 0.25)"
        ).alias(
            "p25_tokens"
        ),

        F.expr(
            f"percentile_approx("
            f"{TOKEN_COUNT_COL}, 0.50)"
        ).alias(
            "median_tokens"
        ),

        F.expr(
            f"percentile_approx("
            f"{TOKEN_COUNT_COL}, 0.75)"
        ).alias(
            "p75_tokens"
        ),

        F.expr(
            f"percentile_approx("
            f"{TOKEN_COUNT_COL}, 0.95)"
        ).alias(
            "p95_tokens"
        ),

        F.max(
            TOKEN_COUNT_COL
        ).alias(
            "max_tokens"
        ),

        F.round(
            F.avg(
                TOKEN_COUNT_COL
            ),
            2,
        ).alias(
            "avg_tokens"
        ),
    )
)

In [0]:
display(
    ticket_length_summary_df
)

##### 23. Inspect Very Short Tickets

In [0]:
display(
    clean_tickets_df
    .filter(
        F.col(TOKEN_COUNT_COL) <= 2
    )
    .select(
        TICKET_ID_COL,
        TEXT_COL,
        CLEAN_TEXT_COL,
        TOKEN_COUNT_COL,
        TARGET_COL,
    )
    .limit(50)
)

##### 24. Inspect Longest Tickets

In [0]:
display(
    clean_tickets_df
    .orderBy(
        F.desc(
            TOKEN_COUNT_COL
        )
    )
    .select(
        TICKET_ID_COL,
        TEXT_COL,
        TOKEN_COUNT_COL,
        TARGET_COL,
    )
    .limit(20)
)

##### 25. Analyze Class Distribution

In [0]:
class_distribution_df = (
    clean_tickets_df
    .groupBy(
        TARGET_COL
    )
    .count()
    .withColumn(
        "percentage",
        F.round(
            (
                F.col("count")
                / F.sum("count").over(
                    Window.partitionBy()
                )
            )
            * 100,
            2,
        ),
    )
    .orderBy(
        F.desc("count")
    )
)

In [0]:
display(
    class_distribution_df
)

##### 26. Verify the Reusable Cleaning Logic

In [0]:
verification_cases = [
    (
        "Internet is DOWN!!!",
        "internet is down",
    ),
    (
        "  Billing   problem  ",
        "billing problem",
    ),
    (
        "Visit https://example.com/help",
        "visit url",
    ),
    (
        "Contact user@example.com please",
        "contact email please",
    ),
]

In [0]:
verification_df = (
    spark.createDataFrame(
        verification_cases,
        [
            "raw_text",
            "expected_text",
        ],
    )
)

In [0]:
verification_df = (
    verification_df
    .withColumn(
        "actual_text",
        clean_text_column(
            F.col("raw_text")
        ),
    )
)

In [0]:
display(
    verification_df
)

In [0]:
failed_verification_count = (
    verification_df
    .filter(
        F.col("expected_text")
        != F.col("actual_text")
    )
    .count()
)

##### 27. Create the Canonical Silver Dataset

In [0]:
nlp_df = (
    clean_tickets_df
    .select(
        TICKET_ID_COL,
        TEXT_COL,
        CLEAN_TEXT_COL,
        TOKENS_COL,
        TOKEN_COUNT_COL,
        TARGET_COL,
    )
)

In [0]:
display(
    nlp_df.limit(20)
)

##### 28. Validate Silver Schema

In [0]:
expected_nlp_columns = {
    TICKET_ID_COL,
    TEXT_COL,
    CLEAN_TEXT_COL,
    TOKENS_COL,
    TOKEN_COUNT_COL,
    TARGET_COL,
}

In [0]:
actual_silver_columns = set(
    nlp_df.columns
)

In [0]:
missing_silver_columns = (
    expected_nlp_columns
    - actual_silver_columns
)

In [0]:
if missing_silver_columns:
    raise ValueError(
        "Silver dataset is missing "
        "required columns: "
        f"{sorted(missing_silver_columns)}"
    )

print(
    "Silver schema validation passed."
)

###### 29. Validate Final Required Fields

In [0]:
final_null_summary_df = (
    nlp_df
    .select(
        F.sum(
            F.col(TICKET_ID_COL)
            .isNull()
            .cast("int")
        ).alias(
            "null_ticket_id"
        ),

        F.sum(
            F.col(CLEAN_TEXT_COL)
            .isNull()
            .cast("int")
        ).alias(
            "null_clean_text"
        ),

        F.sum(
            F.col(TARGET_COL)
            .isNull()
            .cast("int")
        ).alias(
            "null_category"
        ),
    )
)

In [0]:
display(
    final_null_summary_df
)

In [0]:
final_null_counts = (
    final_null_summary_df
    .first()
    .asDict()
)

if any(
    value > 0
    for value in final_null_counts.values()
):
    raise ValueError(
        "Unexpected null values found "
        "in required persisted_nlp_df columns."
    )

print(
    "Final null validation passed."
)

##### 30. Validate Final Row Count

In [0]:
final_row_count = (
    nlp_df.count()
)

print(
    f"Final persisted_nlp_df rows: "
    f"{final_row_count:,}"
)

In [0]:
if final_row_count == 0:
    raise ValueError(
        "Final persisted_nlp_df dataset "
        "contains no records."
    )

In [0]:
total_removed_count = (
    source_row_count
    - final_row_count
)

print(
    f"Rows removed during preprocessing: "
    f"{total_removed_count:,}"
)

##### 31. Persist the nlp_df Dataset

In [0]:
(
    nlp_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        NLP_CLEAN_TABLE
    )
)

In [0]:
print(
    f"Saved nlp table: "
    f"{NLP_CLEAN_TABLE}"
)

##### 32. Verify the Persisted Table

In [0]:
persisted_nlp_df = (
    spark.table(
        NLP_CLEAN_TABLE
    )
)

In [0]:
persisted_row_count = (
    persisted_nlp_df.count()
)

print(
    f"Persisted nlp rows: "
    f"{persisted_row_count:,}"
)

In [0]:
if persisted_row_count != final_row_count:
    raise ValueError(
        "Persisted nlp row count "
        "does not match the prepared dataset."
    )

print(
    "Persisted nlp verification passed."
)

##### 33. Verify Persisted Schema

In [0]:
persisted_columns = set(
    persisted_nlp_df.columns
)

if persisted_columns != expected_nlp_columns:
    raise ValueError(
        "Persisted nlp schema "
        "does not match the expected contract."
    )

print(
    "Persisted nlp schema verification passed."
)

##### 34. Inspect Final Persisted Output

In [0]:
display(
    persisted_nlp_df.limit(20)
)

In [0]:
%sql
select * from dbw_agentic_ai_dev.support_ticket_ai.nlp_preprocessed_tickets

In [0]:
%sql
select * from dbw_agentic_ai_dev.support_ticket_ai.silver_support_tickets

##### 35. Final Data Contract


``` text

ticket_id
    ↓
support-ticket identifier

ticket_text
    ↓
original normalized support-ticket text

clean_text
    ↓
cleaned canonical text

tokens
    ↓
whitespace-tokenized representation

token_count
    ↓
number of tokens in the ticket

category
    ↓
supervised classification target

```

##### 36. Important Training / Serving Principle



There is an important distinction between this nlp_preprocessed_tickets -table preparation and model-serving preprocessing.

Notebook 02 prepares a high-quality reusable NLP dataset:

``` text

Bronze
   ↓
clean_text
   ↓
nlp_preprocessed_tickets

```

Later, when we build the deployable classifier, we should package the preprocessing required by that model together with the fitted model whenever possible.

For example:

``` text

Raw API ticket_text
        ↓
model preprocessing
        ↓
TF-IDF transformation
        ↓
classifier
        ↓
predicted category

```

The caller should ideally send: "My internet keeps disconnecting."

not manually generate TF-IDF features.

This helps prevent training-serving skew.

##### 37. What This Notebook Does Not Do



Notebook 02 does not:

- build vocabulary
- create Bag-of-Words vectors
- calculate TF-IDF
- split train / validation / test data
- train a classifier
- generate embeddings
- train a neural network
- use transformers
- log MLflow experiments
- register models
- deploy endpoints

Those responsibilities belong to later stages.

This separation keeps each notebook focused.

##### 38. Production Design Decisions

This notebook follows these design principles:

- src/project_config.py is the single source of project configuration.
- src/text_preprocessing.py contains reusable preprocessing logic.
- No %run notebook dependency exists.
- Notebook 02 is independently runnable.
- Bronze is treated as the source contract.
- nlp_preprocessed_tickets is treated as the curated NLP data contract.
- Required schema assumptions fail early.
- Invalid training records are explicitly removed.
- Category values are validated against expected labels.
- Duplicates are inspected rather than blindly deleted.
- Text cleaning is deterministic.
- Stop-word removal, stemming, and lemmatization are not applied blindly.
- Cleaning behavior has deterministic verification cases.
- Persisted Delta output is re-read and validated.
- Downstream notebooks depend on persistent tables rather than notebook memory.

##### Key Learnings

- Raw text must be cleaned and normalized before traditional NLP feature extraction.
- Cleaning decisions can change the meaning of text, so preprocessing should be conservative.
- Tokenization converts cleaned text into individual units that can be analyzed.
- Duplicate text matters because it can later cause train/test leakage.
- Target-category validation helps detect label inconsistencies early.
- Token-length analysis prepares us for later traditional NLP and transformer decisions.
- Reusable preprocessing belongs in src, while notebooks orchestrate and explain the workflow.
- Unity Catalog Delta tables provide stable contracts between independently runnable notebooks.
- Preprocessing used for deployed inference should ultimately be packaged with the model whenever possible.

##### Conclusion

Notebook 02 transforms:

dbw_agentic_ai_dev.support_ticket_ai.bronze_support_tickets

into:

dbw_agentic_ai_dev.support_ticket_ai.nlp_preprocessed_tickets

using:

``` text

src/project_config.py
        ↓
shared configuration

src/text_preprocessing.py
        ↓
reusable text cleaning

02_text_cleaning_and_tokenization
        ↓
validation
cleaning
tokenization
quality checks
persistence

```

The nlp_preprocessed_tickets dataset now becomes the stable input for the next stage of the NLP learning path.

##### Next Notebook

03_vocabulary_and_bag_of_words

The next notebook should independently read: nlp_preprocessed_tickets

and then move into the important NLP transition:

``` text

clean text
    ↓
training data split
    ↓
vocabulary
    ↓
CountVectorizer
    ↓
Bag-of-Words sparse vectors

```

